# Phase 4 - P4-E02: Learned Bi-Temporal Change Intelligence (LEVIR-CC Change Description)

## License Gate & Dataset Audit
- `cdvqa_annotation_license`: Apache-2.0
- `second_dataset_access`: public
- `second_image_license_status`: UNRESOLVED
- `cdvqa_full_dataset_license_gate`: BLOCKED

> **Decision**: For the SIH MVP, P4-E02 switches to **CHANGE DESCRIPTION** using **LEVIR-CC** (permitted by the bi-temporal change description/VQA requirement).
>
> **LEVIR-CC Provenance & Terms**:
> - Upstream imagery derives from LEVIR-CD (Hao Chen & Zhenwei Shi, Beihang University).
> - Use is restricted to academic / non-commercial research purposes.
> - Imagery is NOT redistributed in the SatQuery repository.
> - HuggingFace/mirror Apache-2.0 metadata does NOT override upstream terms.
> - **Test Set Policy**: SEALED (evaluation is performed strictly on the validation split).

In [ ]:
# Install dependencies (frozen)
!pip install -q transformers==4.57.6 huggingface-hub==0.36.2 accelerate==1.14.0


In [ ]:
import os
import sys
import subprocess
from pathlib import Path
import torch
import transformers
import huggingface_hub
import accelerate

print('Runtime versions:')
print('  Python:         ', sys.version.split()[0])
print('  torch:          ', torch.__version__)
print('  CUDA available: ', torch.cuda.is_available())
if torch.cuda.is_available():
    print('  CUDA device:    ', torch.cuda.get_device_name(0))
print('  transformers:   ', transformers.__version__)
print('  huggingface_hub:', huggingface_hub.__version__)
print('  accelerate:     ', accelerate.__version__)

# Smoke import test: must succeed BEFORE downloading 2.68 GB LEVIR-CC
print('Performing SmolVLM smoke import...')
from transformers import AutoModelForImageTextToText, AutoProcessor
print('SmolVLM smoke import PASSED.')

print('Initializing Phase 4 E02 LEVIR-CC Change Description Evaluation...')
assert 'SATQUERY_REMOTE_OUTPUT' in os.environ, 'SATQUERY_REMOTE_OUTPUT environment variable missing'
output_dir = Path('/kaggle/working/satquery-output') / os.environ['SATQUERY_REMOTE_OUTPUT']
output_dir.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'HuggingFaceTB/SmolVLM-256M-Instruct'
print(f'Target model ID: {MODEL_ID}')

# Add satquery repo to path
repo_root = Path('/kaggle/working/SATQuery')
if not repo_root.exists():
    repo_url = os.environ.get('SATQUERY_REPO_URL', 'https://github.com/bishuk-dev/SIH-26167-SATQuery.git')
    print(f'Cloning {repo_url}...')
    subprocess.run(['git', 'clone', repo_url, str(repo_root)], check=True)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


In [ ]:
from scripts.kaggle.p4_e02_baseline import run_p4_e02_evaluation

metrics = run_p4_e02_evaluation(output_dir)
status = metrics.get('status')
print('P4-E02 evaluation completed with status:', status)

if status == 'PASS':
    expected_files = ['validation_metrics.json', 'validation_predictions.jsonl', 'runner_meta.json']
    for fname in expected_files:
        fpath = output_dir / fname
        if not fpath.exists():
            raise FileNotFoundError(f'Expected result file missing after evaluation: {fpath}')
        print(f'  Verified: {fpath.name} ({fpath.stat().st_size} bytes)')
    print('Evaluation output directory verified:')
    for p in output_dir.iterdir():
        print(f'  - {p.name} ({p.stat().st_size} bytes)')
else:
    print('Evaluation did not pass. Check evaluation_failure.json for details.')